In [1]:
# Load data to the dataframe as a starting point to create the gold layer
df = spark.read.table("Sales.dbo.sales_silver")

StatementMeta(, d8a9160f-a5bf-4ef7-b494-3ebdba4d6072, 3, Finished, Available, Finished, False)

In [2]:
from pyspark.sql.types import *
from delta.tables import *

# Define the schema for the dimdate_gold table
DeltaTable.createIfNotExists(spark) \
    .tableName("sales.dbo.dimdate_gold") \
    .addColumn("OrderDate", DateType()) \
    .addColumn("Day", IntegerType()) \
    .addColumn("Month", IntegerType()) \
    .addColumn("Year", IntegerType()) \
    .addColumn("mmmyyyy", StringType()) \
    .addColumn("yyyymm", StringType()) \
    .execute()

StatementMeta(, d8a9160f-a5bf-4ef7-b494-3ebdba4d6072, 4, Finished, Available, Finished, False)

In [3]:
from pyspark.sql.functions import col, dayofmonth, month, year, date_format

# Create dataframe for dimDate_gold
dfdimDate_gold = df.dropDuplicates(["OrderDate"]).select(col("OrderDate"), \
        dayofmonth("OrderDate").alias("Day"), \
        month("OrderDate").alias("Month"), \
        year("OrderDate").alias("Year"), \
        date_format(col("OrderDate"), "MMM-yyyy").alias("mmmyyyy"), \
        date_format(col("OrderDate"), "yyyyMM").alias("yyyymm"), \
    ).orderBy("OrderDate")

display(dfdimDate_gold.head(10))

StatementMeta(, d8a9160f-a5bf-4ef7-b494-3ebdba4d6072, 6, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 41a776c4-fa9d-477e-ab63-5df94af37e08)

In [4]:
from delta.tables import *

deltaTable = DeltaTable.forPath(spark, 'Tables/dbo/dimdate_gold')

dfUpdates = dfdimDate_gold

deltaTable.alias('gold') \
  .merge(
    dfUpdates.alias('updates'),
    'gold.OrderDate = updates.OrderDate'
  ) \
   .whenMatchedUpdate(set = {}) \
 .whenNotMatchedInsert(values =
    {
      "OrderDate": "updates.OrderDate",
      "Day": "updates.Day",
      "Month": "updates.Month",
      "Year": "updates.Year",
      "mmmyyyy": "updates.mmmyyyy",
      "yyyymm": "updates.yyyymm"
    }
  ) \
  .execute()

StatementMeta(, d8a9160f-a5bf-4ef7-b494-3ebdba4d6072, 8, Finished, Available, Finished, False)

In [5]:
from pyspark.sql.types import *
from delta.tables import *

# Create customer_gold dimension delta table
DeltaTable.createIfNotExists(spark) \
    .tableName("sales.dbo.dimcustomer_gold") \
    .addColumn("CustomerName", StringType()) \
    .addColumn("Email",  StringType()) \
    .addColumn("First", StringType()) \
    .addColumn("Last", StringType()) \
    .addColumn("CustomerID", LongType()) \
    .execute()

StatementMeta(, d8a9160f-a5bf-4ef7-b494-3ebdba4d6072, 9, Finished, Available, Finished, False)

In [6]:
from pyspark.sql.functions import col, split

# Create customer_silver dataframe
dfdimCustomer_silver = df.dropDuplicates(["CustomerName","Email"]).select(col("CustomerName"),col("Email")) \
    .withColumn("First",split(col("CustomerName"), " ").getItem(0)) \
    .withColumn("Last",split(col("CustomerName"), " ").getItem(1))

display(dfdimCustomer_silver.head(10))

StatementMeta(, d8a9160f-a5bf-4ef7-b494-3ebdba4d6072, 11, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 9973205e-9817-4723-828c-c79d99adc82b)

In [7]:
from pyspark.sql.functions import monotonically_increasing_id, col, when, coalesce, max, lit

dfdimCustomer_temp = spark.read.table("Sales.dbo.dimCustomer_gold")

MAXCustomerID = dfdimCustomer_temp.select(coalesce(max(col("CustomerID")),lit(0)).alias("MAXCustomerID")).first()[0]

dfdimCustomer_gold = dfdimCustomer_silver.join(dfdimCustomer_temp,(dfdimCustomer_silver.CustomerName == dfdimCustomer_temp.CustomerName) & (dfdimCustomer_silver.Email == dfdimCustomer_temp.Email), "left_anti")

dfdimCustomer_gold = dfdimCustomer_gold.withColumn("CustomerID",monotonically_increasing_id() + MAXCustomerID + 1)

display(dfdimCustomer_gold.head(10))

StatementMeta(, d8a9160f-a5bf-4ef7-b494-3ebdba4d6072, 14, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 092486c5-750d-40f3-9d69-d7a28d7e451e)

In [8]:
from delta.tables import *

deltaTable = DeltaTable.forPath(spark, 'Tables/dbo/dimcustomer_gold')

dfUpdates = dfdimCustomer_gold

deltaTable.alias('gold') \
  .merge(
    dfUpdates.alias('updates'),
    'gold.CustomerName = updates.CustomerName AND gold.Email = updates.Email'
  ) \
   .whenMatchedUpdate(set = {}) \
 .whenNotMatchedInsert(values =
    {
      "CustomerName": "updates.CustomerName",
      "Email": "updates.Email",
      "First": "updates.First",
      "Last": "updates.Last",
      "CustomerID": "updates.CustomerID"
    }
  ) \
  .execute()

StatementMeta(, d8a9160f-a5bf-4ef7-b494-3ebdba4d6072, 16, Finished, Available, Finished, False)

In [9]:
from pyspark.sql.types import *
from delta.tables import *

DeltaTable.createIfNotExists(spark) \
    .tableName("sales.dbo.dimproduct_gold") \
    .addColumn("ItemName", StringType()) \
    .addColumn("ItemID", LongType()) \
    .addColumn("ItemInfo", StringType()) \
    .execute()

StatementMeta(, d8a9160f-a5bf-4ef7-b494-3ebdba4d6072, 17, Finished, Available, Finished, False)

In [10]:
from pyspark.sql.functions import col, split, lit, when

dfdimProduct_silver = df.dropDuplicates(["Item"]).select(col("Item")) \
    .withColumn("ItemName",split(col("Item"), ", ").getItem(0)) \
    .withColumn("ItemInfo",when((split(col("Item"), ", ").getItem(1).isNull() | (split(col("Item"), ", ").getItem(1)=="")),lit("")).otherwise(split(col("Item"), ", ").getItem(1)))

display(dfdimProduct_silver.head(10))

StatementMeta(, d8a9160f-a5bf-4ef7-b494-3ebdba4d6072, 19, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, f49d1de6-9997-470d-ad15-096b967158c4)

In [11]:
from pyspark.sql.functions import monotonically_increasing_id, col, lit, max, coalesce

dfdimProduct_temp = spark.read.table("Sales.dbo.dimProduct_gold")

MAXProductID = dfdimProduct_temp.select(coalesce(max(col("ItemID")),lit(0)).alias("MAXItemID")).first()[0]

dfdimProduct_gold = dfdimProduct_silver.join(dfdimProduct_temp,(dfdimProduct_silver.ItemName == dfdimProduct_temp.ItemName) & (dfdimProduct_silver.ItemInfo == dfdimProduct_temp.ItemInfo), "left_anti")

dfdimProduct_gold = dfdimProduct_gold.withColumn("ItemID",monotonically_increasing_id() + MAXProductID + 1)

display(dfdimProduct_gold.head(10))

StatementMeta(, d8a9160f-a5bf-4ef7-b494-3ebdba4d6072, 22, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 76b62a08-42d9-4194-a66c-2b82b5d28e36)

In [12]:
from delta.tables import *

deltaTable = DeltaTable.forPath(spark, 'Tables/dbo/dimproduct_gold')

dfUpdates = dfdimProduct_gold

deltaTable.alias('gold') \
  .merge(
        dfUpdates.alias('updates'),
        'gold.ItemName = updates.ItemName AND gold.ItemInfo = updates.ItemInfo'
        ) \
        .whenMatchedUpdate(set = {}) \
        .whenNotMatchedInsert(values =
         {
          "ItemName": "updates.ItemName",
          "ItemInfo": "updates.ItemInfo",
          "ItemID": "updates.ItemID"
          }
          ) \
          .execute()

StatementMeta(, d8a9160f-a5bf-4ef7-b494-3ebdba4d6072, 24, Finished, Available, Finished, False)

In [13]:
from pyspark.sql.types import *
from delta.tables import *

DeltaTable.createIfNotExists(spark) \
    .tableName("sales.dbo.factsales_gold") \
    .addColumn("CustomerID", LongType()) \
    .addColumn("ItemID", LongType()) \
    .addColumn("OrderDate", DateType()) \
    .addColumn("Quantity", IntegerType()) \
    .addColumn("UnitPrice", FloatType()) \
    .addColumn("Tax", FloatType()) \
    .execute()

StatementMeta(, d8a9160f-a5bf-4ef7-b494-3ebdba4d6072, 25, Finished, Available, Finished, False)

In [14]:
from pyspark.sql.functions import col, split, lit, when

dfdimCustomer_temp = spark.read.table("Sales.dbo.dimCustomer_gold")
dfdimProduct_temp = spark.read.table("Sales.dbo.dimProduct_gold")

df = df.withColumn("ItemName",split(col("Item"), ", ").getItem(0)) \
    .withColumn("ItemInfo",when((split(col("Item"), ", ").getItem(1).isNull() | (split(col("Item"), ", ").getItem(1)=="")),lit("")).otherwise(split(col("Item"), ", ").getItem(1)))

dffactSales_gold = df.alias("df1").join(dfdimCustomer_temp.alias("df2"),(df.CustomerName == dfdimCustomer_temp.CustomerName) & (df.Email == dfdimCustomer_temp.Email), "left") \
        .join(dfdimProduct_temp.alias("df3"),(df.ItemName == dfdimProduct_temp.ItemName) & (df.ItemInfo == dfdimProduct_temp.ItemInfo), "left") \
    .select(col("df2.CustomerID") \
        , col("df3.ItemID") \
        , col("df1.OrderDate") \
        , col("df1.Quantity") \
        , col("df1.UnitPrice") \
        , col("df1.Tax") \
    ).orderBy(col("df1.OrderDate"), col("df2.CustomerID"), col("df3.ItemID"))

display(dffactSales_gold.head(10))

StatementMeta(, d8a9160f-a5bf-4ef7-b494-3ebdba4d6072, 27, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 023e3816-4253-4de9-82a9-a5863669a176)

In [15]:
from delta.tables import *

deltaTable = DeltaTable.forPath(spark, 'Tables/dbo/factsales_gold')

dfUpdates = dffactSales_gold

deltaTable.alias('gold') \
  .merge(
    dfUpdates.alias('updates'),
    'gold.OrderDate = updates.OrderDate AND gold.CustomerID = updates.CustomerID AND gold.ItemID = updates.ItemID'
  ) \
   .whenMatchedUpdate(set = {}) \
 .whenNotMatchedInsert(values =
    {
      "CustomerID": "updates.CustomerID",
      "ItemID": "updates.ItemID",
      "OrderDate": "updates.OrderDate",
      "Quantity": "updates.Quantity",
      "UnitPrice": "updates.UnitPrice",
      "Tax": "updates.Tax"
    }
  ) \
  .execute()

StatementMeta(, d8a9160f-a5bf-4ef7-b494-3ebdba4d6072, 29, Finished, Available, Finished, False)